In [66]:
import datetime as dt
import numpy as np
import pandas as pd
import numpy as np
import tabulate as tb
from typing import Dict
import tensorflow as tf
import re
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler


SEED = 42
np.random.seed(SEED)
CURRENT_DATE = dt.datetime(2025, 6, 1)

In [ ]:
data_players = pd.read_csv('./../data/players_data.csv')
data_players['Month'] = pd.to_datetime(data_players['Month'], format='%Y-%m-%d', errors='raise')


dic_players = {
    f"{row['Player']}_{row['Surface']}_{row['Month'].strftime('%Y-%m')}": {
        'EloAll': row['EloAll'],
        'MatchesAll': row['MatchesAll'],
        'WinRateAll': row['WinRateAll'],
        'EloMonth': row['EloMonth'],
        'MatchesMonth': row['MatchesMonth'],
        'WinRateMonth': row['WinRateMonth'],
    }
    for _, row in data_players.iterrows()
}


data_matches = pd.read_csv('./../data/wta.csv')
data_matches['Date'] = pd.to_datetime(data_matches['Date'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
data_matches['Date'].dropna(inplace=True)
data_matches['Date'] = data_matches['Date'].apply(lambda x: x.replace(day=1))


data_matches['WinnerBinary'] = (data_matches['Winner'] == data_matches['Player_1']).astype(int)

print(tb.tabulate(data_players.sort_values(by='EloAll', ascending=True).head(), headers='keys', tablefmt='psql'))
print(tb.tabulate(data_matches.head(), headers='keys', tablefmt='psql'))

In [ ]:
nn_data = []

for index, row in data_matches.iterrows():
    p1 = row['Player_1']
    p2 = row['Player_2']
    surface = row['Surface']
    data = row['Date'] - pd.DateOffset(months=1)
    
    if pd.isna(data):
        print(f"Skipping match {index} due to invalid date. DATE: {data}")
        continue
    
    p1_data = dic_players.get(f"{p1}_{surface}_{data.strftime('%Y-%m')}", {})
    p2_data = dic_players.get(f"{p2}_{surface}_{data.strftime('%Y-%m')}", {})
    
    nn_data.append({
        'Player_1': p1,
        'Player_2': p2,
        'Surface': surface,
        'Date': row['Date'],
                
        'p1_EloMonth': p1_data.get('EloMonth', 1500),
        'p1_MatchesMonth': p1_data.get('MatchesMonth', 0),
        'p1_WinRateMonth': p1_data.get('WinRateMonth', 0.5),
        
        'p1_EloAll': p1_data.get('EloAll', 1500),
        'p1_MatchesAll': p1_data.get('MatchesAll', 0),
        'p1_WinRateAll': p1_data.get('WinRateAll', 0.5),
        
        'p2_EloMonth': p2_data.get('EloMonth', 1500),
        'p2_MatchesMonth': p2_data.get('MatchesMonth', 0),
        'p2_WinRateMonth': p2_data.get('WinRateMonth', 0.5),
        
        'p2_EloAll': p2_data.get('EloAll', 1500),
        'p2_MatchesAll': p2_data.get('MatchesAll', 0),
        'p2_WinRateAll': p2_data.get('WinRateAll', 0.5),
        
        'Winner': row['WinnerBinary'],
    })
    
data_nn = pd.DataFrame(nn_data)
data_nn = data_nn.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(tb.tabulate(data_nn.head(20), headers='keys', tablefmt='psql'))

Skipping match 9036 due to invalid date. DATE: NaT
Skipping match 13479 due to invalid date. DATE: NaT
Skipping match 13479 due to invalid date. DATE: NaT
+----+---------------+--------------------+-----------+---------------------+---------------+-------------------+-------------------+-------------+-----------------+-----------------+---------------+-------------------+-------------------+-------------+-----------------+-----------------+----------+
|    | Player_1      | Player_2           | Surface   | Date                |   p1_EloMonth |   p1_MatchesMonth |   p1_WinRateMonth |   p1_EloAll |   p1_MatchesAll |   p1_WinRateAll |   p2_EloMonth |   p2_MatchesMonth |   p2_WinRateMonth |   p2_EloAll |   p2_MatchesAll |   p2_WinRateAll |   Winner |
|----+---------------+--------------------+-----------+---------------------+---------------+-------------------+-------------------+-------------+-----------------+-----------------+---------------+-------------------+-------------------+----

In [ ]:
# train, test
data_train = data_nn[data_nn['Date'] < CURRENT_DATE].reset_index(drop=True)
data_test = data_nn[data_nn['Date'] >= CURRENT_DATE].reset_index(drop=True)

X_train, y_train = data_train.drop(columns=['Date', 'Winner', 'Surface', 'Player_1', 'Player_2']), data_train['Winner']
X_test, y_test = data_test.drop(columns=['Date', 'Winner', 'Surface', 'Player_1', 'Player_2']), data_test['Winner']

In [ ]:
columns_to_exclude = ['Rank_1', 'Rank_2']  
columns_to_scale = [X for X in X_train.columns if X not in columns_to_exclude]

# Scale only the selected columns
scaler = StandardScaler()
X_train_scaled_part = scaler.fit_transform(X_train[columns_to_scale])
X_test_scaled_part = scaler.transform(X_test[columns_to_scale])

# Replace the scaled columns in the original data
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[columns_to_scale] = X_train_scaled_part
X_test_scaled[columns_to_scale] = X_test_scaled_part

print(tb.tabulate(X_train_scaled.head(), headers='keys', tablefmt='psql'))

+----+---------------+-------------------+-------------------+-------------+-----------------+-----------------+---------------+-------------------+-------------------+-------------+-----------------+-----------------+
|    |   p1_EloMonth |   p1_MatchesMonth |   p1_WinRateMonth |   p1_EloAll |   p1_MatchesAll |   p1_WinRateAll |   p2_EloMonth |   p2_MatchesMonth |   p2_WinRateMonth |   p2_EloAll |   p2_MatchesAll |   p2_WinRateAll |
|----+---------------+-------------------+-------------------+-------------+-----------------+-----------------+---------------+-------------------+-------------------+-------------+-----------------+-----------------|
|  0 |     -0.123814 |         -0.478541 |         -0.486878 |   -0.636114 |       -0.829817 |       -2.11365  |     -0.120934 |        -0.476242  |         -0.486208 |    2.51899  |        2.62265  |      1.03858    |
|  1 |     -0.123814 |         -0.478541 |         -0.486878 |   -0.636114 |       -0.829817 |       -2.11365  |     -1.0090

In [ ]:
print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Training target distribution: {y_train.value_counts()}")



model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Early stopping to prevent overfitting
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# Train the model
print("Training the model...")
history = model.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

# Evaluate on test set
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\nTest Accuracy: {test_accuracy:.4f}")

# Make predictions
y_pred_proba = model.predict(X_test_scaled)
y_pred = (y_pred_proba > 0.5).astype(int)

# Calculate additional metrics
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Training data shape: (42513, 12)
Test data shape: (95, 12)
Training target distribution: Winner
1    21257
0    21256
Name: count, dtype: int64
Training the model...
Epoch 1/100


c:\Personal\TennisPredictor\.venv\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1063/1063 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.6199 - loss: 0.6518 - val_accuracy: 0.6290 - val_loss: 0.6368
Epoch 2/100
1063/1063 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6330 - loss: 0.6386 - val_accuracy: 0.6285 - val_loss: 0.6374
Epoch 3/100
1063/1063 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6375 - loss: 0.6357 - val_accuracy: 0.6307 - val_loss: 0.6360
Epoch 4/100
1063/1063 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6398 - loss: 0.6369 - val_accuracy: 0.6324 - val_loss: 0.6371
Epoch 5/100
1063/1063 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6421 - loss: 0.6356 - val_accuracy: 0.6322 - val_loss: 0.6362
Epoch 6/100
1063/1063 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6358 - loss: 0.6382 - val_accuracy: 0.6307 - val_loss: 0.6356
Epoch 7/100
1063/1063 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6367 - loss: 0.6353 - val_accuracy: 0.6317 - val_loss: 0.6361
Epoch 8/100
1063/1063 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6396 - loss: 0.6331 - val_